# Algoritmos de Busca

### Disciplina de Sistemas Inteligentes

### Departamento de Informática e Estatística

### Profa. Nathalia da Cruz Alves


**Nota 1**: Em caso de reprodução, publicação ou compartilhamento deste material, a autoria deverá ser obrigatoriamente mantida.

**Nota 2**: Este material foi elaborado com o auxílio de ferramentas de Inteligência Artificial (IA), utilizadas para o refinamento de explicações e revisão de trechos de código. A autoria e a responsabilidade pelo conteúdo final permanecem com a autora.

Estudante(s): insira seu nome aqui


**Observação**: Solicita-se que sejam informados apenas os nomes dos estudantes efetivamente participantes da atividade.


Neste notebook, vamos explorar diferentes algoritmos de busca em Inteligência Artificial utilizando um grafo simplificado com cidades do estado de Santa Catarina.

Os algoritmos que vamos implementar e comparar são:
1. **Busca em Largura (BFS)**: Não informada (menor número de passos)
2. **Busca em Profundidade (DFS)**: Não informada (menor uso de memória)
3. **Busca de Custo Uniforme (UCS)**: Baseada em custos reais (distâncias em km)
4. **Busca de Melhor Escolha (GBF)**: Baseada puramente na heurística

## Configurações iniciais

### Bibliotecas utilizadas

* **`folium`**: Cria o mapa interativo baseado no OpenStreetMap. Permite plotar as cidades de Santa Catarina usando coordenadas reais (latitude e longitude) e desenhar as rotas encontradas pelos algoritmos de forma visual e navegável.
* **`networkx`**: Facilita a modelagem e a manipulação do grafo (o mapa em si, contendo as cidades como vértices e as estradas/distâncias como arestas).
* **`matplotlib.pyplot`**: Utilizada para renderizar gráficos e visualizações estáticas complementares, caso seja necessário desenhar o grafo diretamente em uma janela gráfica do Python.
* **`collections.deque`**: Fornece a estrutura de dados de **Fila (Queue)** eficiente (com operações rápidas nas pontas), essencial para implementar a **Busca em Largura (BFS)** seguindo a lógica FIFO (*First-In, First-Out*).
* **`heapq`**: Fornece algoritmos para implementar uma **Fila de Prioridade (Priority Queue)** em Python. É fundamental para os algoritmos que ordenam os nós por menor custo ou heurística (**UCS**, **GBF** e **A***).

In [ ]:
!pip install folium

In [ ]:
import folium
import networkx as nx
import matplotlib.pyplot as plt
from collections import deque
import heapq

## Definindo o grafo de SC (Cidades, Conexões e Distâncias em km aproximadas)

* **`mapa_sc`**: Dicionário que representa o grafo. Cada cidade aponta para seus vizinhos e para o custo (distância real em km) da aresta.
* **`coordenadas_sc`**: Dicionário com latitude e longitude reais de cada cidade, usado para desenhar o mapa interativo no `folium`.
* **`heuristica_fln`**: Valores de heurística ($h$) de cada cidade até o objetivo final (**Florianópolis**), estimados em linha reta para serem usados nas buscas informadas (GBF e A*).

In [ ]:
# Formato: 'Cidade': { 'Vizinho': Custo_Distancia }
mapa_sc = {
    'Florianópolis': {'São José': 15, 'Itajaí': 100},
    'São José': {'Florianópolis': 15, 'Palhoça': 10, 'Biguaçu': 15},
    'Palhoça': {'São José': 10, 'Lages': 220},
    'Biguaçu': {'São José': 15, 'Blumenau': 120},
    'Itajaí': {'Florianópolis': 100, 'Blumenau': 50, 'Joinville': 85},
    'Blumenau': {'Biguaçu': 120, 'Itajaí': 50, 'Joinville': 130, 'Lages': 240},
    'Joinville': {'Itajaí': 85, 'Blumenau': 130, 'Chapecó': 500},
    'Lages': {'Palhoça': 220, 'Blumenau': 240, 'Chapecó': 380},
    'Chapecó': {'Lages': 380, 'Joinville': 500}
}

# Coordenadas reais (Latitude, Longitude) das cidades de SC
coordenadas_sc = {
    'Florianópolis': (-27.5954, -48.5480),
    'São José': (-27.6136, -48.6366),
    'Palhoça': (-27.6447, -48.6703),
    'Biguaçu': (-27.4950, -48.6561),
    'Itajaí': (-26.9078, -48.6619),
    'Blumenau': (-26.9194, -49.0661),
    'Joinville': (-26.3045, -48.8487),
    'Lages': (-27.8159, -50.3264),
    'Chapecó': (-27.1004, -52.6156)
}

# Heurística da distância em linha reta (valores fixos h) para o objetivo 'Florianópolis'
heuristica_fln = {
    'Florianópolis': 0,
    'São José': 12,
    'Palhoça': 20,
    'Biguaçu': 18,
    'Itajaí': 85,
    'Blumenau': 105,
    'Joinville': 160,
    'Lages': 190,
    'Chapecó': 420
}

In [ ]:
# @title Função que desenha o grafo: desenhar_mapa(caminho_destacado=None)

def desenhar_mapa(caminho_destacado=None):
    plt.figure(figsize=(10, 7))
    G = nx.Graph()

    # Adicionando arestas e pesos
    for cidade, vizinhos in mapa_sc.items():
        for vizinho, custo in vizinhos.items():
            G.add_edge(cidade, vizinho, weight=custo)

    pos = nx.spring_layout(G, seed=42) # Layout fixo para consistência

    # Cores padrão
    node_colors = ['lightblue' for _ in G.nodes()]

    # Se houver um caminho para destacar
    edge_colors = 'gray'
    if caminho_destacado:
        path_edges = list(zip(caminho_destacado[:-1], caminho_destacado[1:]))
        node_colors = ['orange' if node in caminho_destacado else 'lightblue' for node in G.nodes()]
        # Destacar nó inicial e final
        node_colors[list(G.nodes()).index(caminho_destacado[0])] = 'green'
        node_colors[list(G.nodes()).index(caminho_destacado[-1])] = 'red'

    nx.draw(G, pos, with_labels=True, node_color=node_colors, node_size=2500, font_size=10, font_weight='bold')
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)

    plt.title("Mapa Rodoviário Simplificado de Santa Catarina", fontsize=14)
    plt.show()

# Visualizar o mapa geral
desenhar_mapa()

In [ ]:
# @title Função que desenha o mapa real: desenhar_mapa_real(caminho_destacado=None)

def desenhar_mapa_real(caminho_destacado=None):
    # Criando o mapa centrado em Santa Catarina
    mapa_sc_real = folium.Map(location=[-27.2423, -50.2189], zoom_start=8)

    # Desenhando as conexões (estradas) em cinza claro
    arestas_desenhadas = set()
    for cidade, vizinhos in mapa_sc.items():
        for vizinho in vizinhos:
            par = tuple(sorted([cidade, vizinho]))
            if par not in arestas_desenhadas:
                arestas_desenhadas.add(par)
                coord1 = coordenadas_sc[cidade]
                coord2 = coordenadas_sc[vizinho]
                folium.PolyLine([coord1, coord2], color='gray', weight=2, opacity=0.6).add_to(mapa_sc_real)

    # Se houver um caminho encontrado pelo algoritmo, destacá-lo em vermelho/azul
    if caminho_destacado:
        coords_caminho = [coordenadas_sc[cidade] for cidade in caminho_destacado]
        folium.PolyLine(coords_caminho, color='red', weight=5, opacity=0.8, tooltip='Rota Encontrada').add_to(mapa_sc_real)

    # Adicionando os marcadores das cidades
    for cidade, coord in coordenadas_sc.items():
        # Cor do marcador baseada no caminho
        cor = 'blue'
        if caminho_destacado:
            if cidade == caminho_destacado[0]:
                cor = 'green'  # Origem
            elif cidade == caminho_destacado[-1]:
                cor = 'darkred'  # Destino
            elif cidade in caminho_destacado:
                cor = 'orange'  # Cidades no meio do caminho

        folium.Marker(
            location=coord,
            popup=f"<b>{cidade}</b>",
            tooltip=cidade,
            icon=folium.Icon(color=cor, icon='info-sign')
        ).add_to(mapa_sc_real)


    return mapa_sc_real

desenhar_mapa_real()

## Implementação dos Algoritmos de Busca

### BFS - Busca em Largura (Breadth-First Search)



In [ ]:
def busca_largura(inicio, objetivo):
    """Busca em Largura (BFS): Explora o grafo por níveis, garantindo
    encontrar o caminho com o menor número de passos (arestas).
    """

    # Se a origem já é o objetivo, retorna imediatamente
    if inicio == objetivo:
        return [inicio]

    # Fronteira armazena caminhos (FIFO) e visitados evita ciclos
    fronteira = deque([[inicio]])
    visitados = set()

    while fronteira:
        caminho = fronteira.popleft()
        atual = caminho[-1]

        # O nó é "visitado" (expandido) quando sai da fronteira
        if atual not in visitados:
            visitados.add(atual)

            if atual == objetivo:
                return caminho

            for vizinho in mapa_sc.get(atual, {}):
                if vizinho not in visitados:
                    nova_rota = list(caminho) + [vizinho]
                    fronteira.append(nova_rota)

    return None

### DFS - Busca em Profundidade (Depth-First Search)

In [ ]:
def busca_profundidade(inicio, objetivo):
    """Busca em Profundidade (DFS): explora o mais fundo possível
    antes de retroceder.
    """

    # Se a origem já é o objetivo, retorna imediatamente
    if inicio == objetivo:
        return [inicio]

    # Fronteira armazena caminhos (LIFO) e visitados evita ciclos
    fronteira = [[inicio]]
    visitados = set()

    while fronteira:
        caminho = fronteira.pop()
        atual = caminho[-1]

        # O nó é "visitado" (expandido) quando sai da fronteira
        if atual not in visitados:
            visitados.add(atual)

            if atual == objetivo:
                return caminho

            for vizinho in mapa_sc.get(atual, {}):
                if vizinho not in visitados:
                    nova_rota = list(caminho) + [vizinho]
                    fronteira.append(nova_rota)

    return None

### UCS - Busca de Custo Uniforme (Uniform Cost Search)

In [ ]:
def busca_custo_uniforme(inicio, objetivo):
    """Busca de Custo Uniforme (UCS): expande sempre o nó de menor
    custo acumulado.
    """

    if inicio == objetivo:
        return [inicio], 0

    # Fronteira armazena tuplas: (custo_total, caminho)
    fronteira = [(0, [inicio])]
    visitados = {}  # Guarda o menor custo encontrado para cada nó

    while fronteira:
        custo, caminho = heapq.heappop(fronteira)
        atual = caminho[-1]

        # O nó é expandido se ainda não foi visitado ou se achamos um caminho mais barato
        if atual not in visitados or custo < visitados[atual]:
            visitados[atual] = custo

            if atual == objetivo:
                return caminho, custo

            for vizinho, peso in mapa_sc.get(atual, {}).items():
                novo_custo = custo + peso

                # Se o vizinho ainda não foi visitado ou encontramos um caminho de menor custo até ele
                if vizinho not in visitados or novo_custo < visitados[vizinho]:
                    novo_caminho = list(caminho) + [vizinho]
                    heapq.heappush(fronteira, (novo_custo, novo_caminho))

    return None, float('inf')

### GBF - Busca de Melhor Escolha (Greedy Best-First)

In [ ]:
def busca_melhor_escolha(inicio, objetivo):
    """Busca de Melhor Escolha (Greedy Best-First): prioriza o nó com menor heurística."""

    if inicio == objetivo:
        return [inicio], 0

    # Fronteira armazena tuplas: (heuristica, caminho)
    h_inicial = heuristica_fln.get(inicio, 0)
    fronteira = [(h_inicial, [inicio])]
    visitados = set()

    while fronteira:
        h, caminho = heapq.heappop(fronteira)
        atual = caminho[-1]

        # O nó é expandido ao sair da fronteira
        if atual not in visitados:
            visitados.add(atual)

            if atual == objetivo:
                # Calcula o custo real do caminho encontrado
                custo_total = sum(mapa_sc[caminho[i]][caminho[i+1]] for i in range(len(caminho) - 1))
                return caminho, custo_total

            for vizinho in mapa_sc.get(atual, {}):
                if vizinho not in visitados:
                    h_vizinho = heuristica_fln.get(vizinho, 0)
                    nova_rota = list(caminho) + [vizinho]
                    heapq.heappush(fronteira, (h_vizinho, nova_rota))

    return None, float('inf')

## Exercícios

### Exercício 1 - Teste de Rotas
Teste as rotas com os diferentes algoritmos para sair de Chapecó a Florianópolis

Responda para cada teste:
1. Qual foi o caminho encontrado?
2. Qual foi o custo total (em km)?

Ao final de todos os testes, responda:

3. Qual foi o algoritmo que encontrou o caminho de menor custo?

In [ ]:
origem = 'Chapecó'
objetivo = 'Florianópolis'
print(f"Testando rotas de {origem} até {objetivo}")

#### BFS

In [ ]:
caminho_bfs = busca_largura(origem, objetivo)
print(f"BFS (Largura): {caminho_bfs}")
desenhar_mapa(caminho_bfs)
desenhar_mapa_real(caminho_bfs)

#### DFS

In [ ]:
caminho_dfs = busca_profundidade(origem, objetivo)
print(f"DFS (Profundidade): {caminho_dfs}")
desenhar_mapa(caminho_dfs)
desenhar_mapa_real(caminho_dfs)

#### UCS

In [ ]:
caminho_ucs, custo_ucs = busca_custo_uniforme(origem, objetivo)
print(f"Custo Uniforme (UCS): {caminho_ucs} | Custo Total: {custo_ucs} km")
desenhar_mapa(caminho_ucs)
desenhar_mapa_real(caminho_ucs)

#### GBF

In [ ]:
caminho_gbf, custo_gbf = busca_melhor_escolha(origem, objetivo)
print(f"Custo Uniforme (GBF): {caminho_gbf} | Custo Total: {custo_gbf} km")
desenhar_mapa(caminho_gbf)
desenhar_mapa_real(caminho_gbf)

### Exercício 2 - Expandindo o Mapa de SC

Para tornar o grafo mais realista e desafiador, você deverá adicionar pelo menos 3 novas cidades de Santa Catarina ao mapa (por exemplo: Criciúma, Dionísio Cerqueira, Balneário Camboriú ou Blumenau conectada a novas regiões).

O que você precisa fazer:
- Atualizar o mapa_sc: Insira as novas cidades com suas respectivas conexões (arestas) e os custos reais (distâncias em km).

- Atualizar as coordenadas_sc: Adicione a latitude e longitude aproximadas das novas cidades.

- Atualizar a heuristica_fln: Defina a estimativa da distância em linha reta (h) de cada nova cidade até Florianópolis.

Escolha uma nova cidade de sua preferência como início e execute todos os algoritmos para chegar até Florianópolis.

Para cada algoritmo, responda:
1. Qual foi o caminho encontrado?

2. Qual foi o custo total (em km)?

Ao final, responda:

3. Quais algoritmos garantiram o menor custo real?